# Kaggle step 1 · Pack the tiles into one zip on Google Drive (run on Colab, CPU)

Kaggle cannot read Google Drive, so this copies the training tiles of every event in
`configs/events.yaml` into a single `tiles.zip` in `MyDrive/kerala_hazard_vision_ai/`.
Runtime: CPU is enough. Takes about 10–20 minutes.

**After it finishes:** in Google Drive, right-click `tiles.zip` → Share → General access:
**Anyone with the link** → Copy link. Paste that link into the Kaggle notebook.

In [ ]:
from google.colab import drive

REPO_URL = "https://github.com/muhammedshahidsds25-collab/kerala-hazard-vision-ai.git"
REPO_DIR = "/content/kerala-hazard-vision-ai"

drive.mount("/content/drive")
!git clone -q {REPO_URL} {REPO_DIR} || git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q .

In [ ]:
import shutil
import zipfile
from pathlib import Path

from kerala_hazard_vision_ai.config import load_config
from kerala_hazard_vision_ai.data.dataset import list_tiles

cfg = load_config("configs/default.yaml")
DRIVE_ROOT = Path("/content/drive/MyDrive") / cfg.data.drive_folder
tiles = list_tiles(DRIVE_ROOT / "tiles", (e.name for e in cfg.events))
print(f"{len(tiles)} tiles from {len(cfg.events)} events")

# Build on the fast local disk, then copy once to Drive. The .npz files are already
# compressed, so store them as they are (ZIP_STORED) instead of compressing twice.
local_zip = Path("/content/tiles.zip")
with zipfile.ZipFile(local_zip, "w", compression=zipfile.ZIP_STORED) as archive:
    for path in tiles:
        archive.write(path, arcname=path.name)
shutil.copy(local_zip, DRIVE_ROOT / "tiles.zip")
print(f"Wrote {DRIVE_ROOT / 'tiles.zip'} ({local_zip.stat().st_size / 1e9:.2f} GB)")